In [0]:
# ============================================================
# CELLULE 0 — Chemins et constantes
# ============================================================
# Règle : toujours définir les chemins EN PREMIER
# Si un chemin change, on ne modifie qu'ici

PATH_FEATURES = "/Volumes/workspace/default/bronze/silver/sensor_features/"
PATH_REPORTS  = "/Volumes/workspace/default/bronze/silver/quality_reports/"

# SLA : seuil minimum accepté en production
SLA_TARGET     = 0.95   # 95%
REPORT_VERSION = "v1.0"

In [0]:
# ============================================================
# CELLULE 1 — Imports
# ============================================================
from pyspark.sql import functions as F
from datetime import datetime, timezone
import json

print("✅ Imports OK")

✅ Imports OK


In [0]:
# ============================================================
# CELLULE 2 — Lecture Silver enrichi (PATH_FEATURES)
# ============================================================
df = spark.read.parquet(PATH_FEATURES)
total_rows = df.count()
total_cols = len(df.columns)

print(f"📊 Données chargées : {total_rows} lignes × {total_cols} colonnes")
print(f"📋 Colonnes : {df.columns}")

📊 Données chargées : 10000 lignes × 26 colonnes
📋 Colonnes : ['sensor_id', 'product_id', 'station_type', 'air_temp_k', 'process_temp_k', 'rotation_speed_rpm', 'torque_nm', 'tool_wear_min', 'machine_failure', 'tool_wear_failure', 'heat_dissipation_failure', 'power_failure', 'overstrain_failure', 'random_failure', 'timestamp', 'rolling_mean_temp_1h', 'rolling_std_temp_1h', 'rolling_mean_rotation_1h', 'rolling_std_rotation_1h', 'rolling_mean_torque_1h', 'rolling_mean_torque_3h', 'temp_diff', 'power_estimated_w', 'tool_wear_rate', 'torque_speed_ratio', 'failure_type']


In [0]:
# ============================================================
# CELLULE 3 — Dimension 1 : Complétude (Completeness)
# ============================================================
# Objectif : mesurer le % de valeurs NON nulles par colonne
# En entreprise, chaque colonne critique a son propre seuil

null_counts = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
]).collect()[0].asDict()

# Calcul du % de complétude par colonne
completeness_by_col = {
    col: round((total_rows - null_count) / total_rows, 4)
    for col, null_count in null_counts.items()
}

# Score global = moyenne de toutes les colonnes
completeness_score = round(
    sum(completeness_by_col.values()) / len(completeness_by_col), 4
)

# Affichage lisible
print("─" * 50)
print(f"{'Colonne':<35} {'Complétude':>10}")
print("─" * 50)
for col, score in completeness_by_col.items():
    flag = "⚠️" if score < SLA_TARGET else "✅"
    print(f"{flag} {col:<33} {score*100:>9.2f}%")
print("─" * 50)
print(f"\n📊 Score Complétude global : {completeness_score*100:.2f}%")

──────────────────────────────────────────────────
Colonne                             Complétude
──────────────────────────────────────────────────
✅ sensor_id                            100.00%
✅ product_id                           100.00%
✅ station_type                         100.00%
✅ air_temp_k                           100.00%
✅ process_temp_k                       100.00%
✅ rotation_speed_rpm                   100.00%
✅ torque_nm                            100.00%
✅ tool_wear_min                        100.00%
✅ machine_failure                      100.00%
✅ tool_wear_failure                    100.00%
✅ heat_dissipation_failure             100.00%
✅ power_failure                        100.00%
✅ overstrain_failure                   100.00%
✅ random_failure                       100.00%
✅ timestamp                            100.00%
✅ rolling_mean_temp_1h                 100.00%
✅ rolling_std_temp_1h                   99.99%
✅ rolling_mean_rotation_1h             100.00%
✅ rol

In [0]:
# ============================================================
# CELLULE 4 — Dimension 2 : Validité (règles métier)
# ============================================================
# On revalide les règles physiques définies en J5
# C'est la cohérence métier, pas juste l'absence de nulls

invalid_rows = df.filter(
    ~(
        F.col("air_temp_k").between(250, 400)           &
        F.col("process_temp_k").between(250, 400)       &
        F.col("rotation_speed_rpm").between(100, 5000)  &
        F.col("torque_nm").between(0, 200)              &
        F.col("tool_wear_min").between(0, 500)
    )
).count()

validity_score = round((total_rows - invalid_rows) / total_rows, 4)

print(f"❌ Lignes invalides détectées : {invalid_rows}")
print(f"📊 Score Validité : {validity_score*100:.2f}%")

❌ Lignes invalides détectées : 0
📊 Score Validité : 100.00%


In [0]:
# ============================================================
# CELLULE 5 — Dimension 3 : Unicité (doublons)
# ============================================================
# sensor_id doit être unique : chaque mesure = un instant unique

distinct_ids   = df.select("sensor_id").distinct().count()
uniqueness_score = round(distinct_ids / total_rows, 4)

duplicates = total_rows - distinct_ids

print(f"🔁 Doublons détectés sur sensor_id : {duplicates}")
print(f"📊 Score Unicité : {uniqueness_score*100:.2f}%")


🔁 Doublons détectés sur sensor_id : 0
📊 Score Unicité : 100.00%


In [0]:
# ============================================================
# CELLULE 6 — Dimension 4 : Cohérence physique
# ============================================================
# Règle physique OCP : la température process > température air
# (le process chauffe toujours plus que l'ambiant)
# Si cette règle est violée → anomalie capteur ou erreur ETL

incoherent_temp = df.filter(
    F.col("process_temp_k") <= F.col("air_temp_k")
).count()

consistency_score = round((total_rows - incoherent_temp) / total_rows, 4)

print(f"⚠️  Lignes incohérentes (process_temp <= air_temp) : {incoherent_temp}")
print(f"📊 Score Cohérence : {consistency_score*100:.2f}%")

⚠️  Lignes incohérentes (process_temp <= air_temp) : 0
📊 Score Cohérence : 100.00%


In [0]:
# ============================================================
# CELLULE 7 — Dimension 5 : Fraîcheur (Timeliness)
# ============================================================
# Les timestamps synthétiques doivent être dans la plage 2020–2024

out_of_range = df.filter(
    ~F.col("timestamp").between("2020-01-01", "2024-12-31")
).count()

timeliness_score = round((total_rows - out_of_range) / total_rows, 4)

print(f"⏰ Lignes hors plage temporelle : {out_of_range}")
print(f"📊 Score Fraîcheur : {timeliness_score*100:.2f}%")

⏰ Lignes hors plage temporelle : 0
📊 Score Fraîcheur : 100.00%


In [0]:
# ============================================================
# CELLULE 8 — Score global SLA + Verdict
# ============================================================
# Pondération professionnelle des 5 dimensions
# La validité et la complétude sont prioritaires pour OCP

weights = {
    "completeness":  0.30,   # 30% — colonnes sans nulls
    "validity":      0.30,   # 30% — règles physiques respectées
    "uniqueness":    0.20,   # 20% — pas de doublons
    "consistency":   0.15,   # 15% — cohérence inter-colonnes
    "timeliness":    0.05    # 5%  — fraîcheur (synthétique ici)
}

scores = {
    "completeness": completeness_score,
    "validity":     validity_score,
    "uniqueness":   uniqueness_score,
    "consistency":  consistency_score,
    "timeliness":   timeliness_score
}

# Score pondéré
global_score = round(
    sum(scores[dim] * weights[dim] for dim in weights), 4
)

sla_passed = global_score >= SLA_TARGET

# Affichage du rapport synthétique
print("=" * 55)
print("       RAPPORT QUALITÉ SILVER — OCP PIPELINE")
print("=" * 55)
for dim, score in scores.items():
    status = "✅" if score >= SLA_TARGET else "❌"
    weight = weights[dim]
    print(f"{status} {dim:<15} {score*100:>6.2f}%   (poids {weight*100:.0f}%)")
print("-" * 55)
print(f"{'SCORE GLOBAL SLA':<22} {global_score*100:>6.2f}%")
print(f"{'SEUIL REQUIS':<22} {SLA_TARGET*100:>6.2f}%")
print("=" * 55)

if sla_passed:
    print("✅ SLA ATTEINT — Silver certifié pour production")
else:
    print("❌ SLA NON ATTEINT — Révision ETL requise")

       RAPPORT QUALITÉ SILVER — OCP PIPELINE
✅ completeness    100.00%   (poids 30%)
✅ validity        100.00%   (poids 30%)
✅ uniqueness      100.00%   (poids 20%)
✅ consistency     100.00%   (poids 15%)
✅ timeliness      100.00%   (poids 5%)
-------------------------------------------------------
SCORE GLOBAL SLA       100.00%
SEUIL REQUIS            95.00%
✅ SLA ATTEINT — Silver certifié pour production


In [0]:
# ============================================================
# CELLULE 9 — Construction du rapport JSON
# ============================================================
# JSON = format universel lisible par tous les outils
# Ce rapport peut être ingéré par Azure Monitor, Power BI, etc.

report = {
    "metadata": {
        "report_version":   REPORT_VERSION,
        "pipeline_name":    "OCP Slurry Pipeline — Silver Quality",
        "generated_at":     datetime.now(timezone.utc).isoformat(),
        "source_path":      PATH_FEATURES,
        "sla_target":       SLA_TARGET
    },
    "dataset_stats": {
        "total_rows":       total_rows,
        "total_columns":    total_cols,
        "invalid_rows":     invalid_rows,
        "duplicate_rows":   duplicates,
        "incoherent_rows":  incoherent_temp
    },
    "dimension_scores": scores,
    "weights":           weights,
    "global_sla_score":  global_score,
    "sla_passed":        sla_passed,
    "completeness_by_column": completeness_by_col
}

print("✅ Rapport JSON construit")
print(json.dumps(report["metadata"], indent=2))

✅ Rapport JSON construit
{
  "report_version": "v1.0",
  "pipeline_name": "OCP Slurry Pipeline \u2014 Silver Quality",
  "generated_at": "2026-05-10T12:24:47.813469+00:00",
  "source_path": "/Volumes/workspace/default/bronze/silver/sensor_features/",
  "sla_target": 0.95
}


In [0]:
# ============================================================
# CELLULE 10 — Export quality_report.json → ADLS
# ============================================================
# On écrit le JSON via Spark pour rester dans le pattern Serverless
# dbutils.fs.put = écriture directe d'un fichier texte dans ADLS

report_json_str = json.dumps(report, indent=2)

# Chemin du fichier JSON dans ADLS
json_path = PATH_REPORTS + "quality_report_silver.json"

# Écriture via dbutils (disponible nativement en Databricks)
dbutils.fs.put(json_path, report_json_str, overwrite=True)

print(f"✅ Rapport exporté → {json_path}")

Wrote 1540 bytes.
✅ Rapport exporté → /Volumes/workspace/default/bronze/silver/quality_reports/quality_report_silver.json


In [0]:
# ============================================================
# CELLULE 11 — Vérification robuste (méthode production)
# ============================================================
# dbutils.fs.head est limité → pour un fichier texte complet,
# on lit via Spark comme un fichier texte brut, puis on joint

lines   = dbutils.fs.head(json_path, 100_000)  # 100k octets — safe
parsed  = json.loads(lines)

print("✅ JSON relu depuis ADLS")
print(f"   → SLA atteint    : {parsed['sla_passed']}")
print(f"   → Score global   : {parsed['global_sla_score']*100:.2f}%")
print(f"   → Généré le      : {parsed['metadata']['generated_at']}")
print(f"   → Lignes total   : {parsed['dataset_stats']['total_rows']}")

✅ JSON relu depuis ADLS
   → SLA atteint    : True
   → Score global   : 100.00%
   → Généré le      : 2026-05-10T12:24:47.813469+00:00
   → Lignes total   : 10000
